# 任务 01 演示：D1 × E1 —— 不同 $p_i$ 的独立 Bernoulli 随机化 + 严格多数暴露

**业务问题。** 公司给一批销售员随机开放 AI 助手。销售员之间会共享客户线索，因此一个人的业绩既可能受自己是否有 AI 影响，也可能受合作同事是否大量使用 AI 影响。Identification 先回答：**在当前实验设计和假设下，这些因果量能否被总体观测分布唯一确定？**

**本任务 = 任务 01**：随机化设计取 D1（每人不同 $p_i$ 的独立 Bernoulli），exposure 取 E1（严格多数，与 shared baseline 相同）。

演示顺序：**业务问题 → setting → support → POINT 推导 → IdentificationResult → 式 (3) 数值验证 → Identification 输出什么**。

## 1. Setting

| 组件 | 定义 |
|---|---|
| 处理前结构 | 网络 $G=(V,A)$（ER 图，$N=300$）与 $X$ 在 assignment 前固定 |
| Assignment（**D1，本任务的改动**） | $T_i\sim\mathrm{Bernoulli}(p_i)$，跨个体独立，$0<p_i<1$，$p_i$ 实验前已知 |
| Exposure（E1，同 baseline） | $K_i(t)=\sum_{j\in N(i)}t_j$，$S_i(t)=\mathbb{1}\{K_i(t)>\lfloor d_i/2\rfloor\}$（严格多数；正好一半归入 $S=0$） |
| Target | $\mu_i(t,s)=\mathbb{E}[Y_i((t,T_{-i}))\mid S_i((t,T_{-i}))=s,A,X]$ |

与 shared baseline 的唯一区别：随机化从"同一个 $1/2$"变成"每人已知的 $p_i$"。邻居的 $p_j$ 不同，$K_i$ 不再是普通 Binomial，而是 **Poisson Binomial**。

## 2. Support：$\pi_i(t,s)$ 是怎么来的

D1 下 $T_i$ 与 $T_{-i}$ 独立，所以

$$\pi_i(t,s)=P(T_i=t,S_i\mid A,X)=P(T_i=t)\,P(S_i=s)=p_i^{\,t}(1-p_i)^{1-t}\,q_i(s),$$

其中 $K_i=\sum_{j\in N(i)}T_j$ 是"每个邻居成功概率都不同"的独立 Bernoulli 之和（Poisson Binomial）：

$$P(K_i=k)=\sum_{A\subseteq N(i),\,|A|=k}\prod_{j\in A}p_j\prod_{j\notin A}(1-p_j),\qquad
q_i(1)=P\big(K_i>\lfloor d_i/2\rfloor\big),\;\;q_i(0)=1-q_i(1).$$

代码用 $O(d_i^2)$ 动态规划算这个 PMF。**哪些状态可能为 0**：$t$ 两值恒可达（$0<p_i<1$）；$s=1$ 要求 $d_i\ge1$（孤立点 $K_i\equiv0$，永远不过半）；$s=0$ 恒可达。

## 3. 为什么有 support 的 arm 是 POINT

对 $\pi_i(t,s)>0$：

$$\begin{aligned}
\mathbb{E}\big[Y_i^{obs}\mid A,X,T_i=t,S_i=s\big]
&=\sum_{t_{-i}:\,S_i((t,t_{-i}))=s}\mathbb{E}\big[Y_i^{obs}\mid A,X,T=(t,t_{-i})\big]\,w(t_{-i})\\
&=\sum_{t_{-i}:\,S_i((t,t_{-i}))=s}\mathbb{E}\big[Y_i((t,t_{-i}))\mid A,X\big]\,w(t_{-i})\\
&=\mu_i(t,s)=\mathbb{E}\big[Y_i^{obs}\mid A,X,T_i=t,S_i=s\big],
\end{aligned}$$

第一行是全概率公式；第二行用 consistency（A2）+ full-vector randomization（A3）；第三行即 target 定义。右侧只依赖总体观测分布 $P_M(T,Y^{obs}\mid A,X)$，故两个观测分布相同的世界必得同一答案 —— POINT。若 $\pi_i(t,s)=0$，条件事件概率为零、条件均值未定义，必须拒绝查询。

## 4. 式 (3) 的数值验证（小图模拟）

上面只验证了数学推导；这里在一个 $N=30$ 的小图上用模拟验证恒等式本身：自构一个**只依赖 $(i,t,\text{邻居处理})$** 的潜在结果模型，比较

- "真值" $\mu_i(t,s)$：按 D1 设计采样 $T_{-i}$、保留 $S_i=s$ 后对 $Y_i((t,T_{-i}))$ 取平均；
- 观测侧 $\mathbb{E}[Y^{obs}_i\mid T_i=t,S_i=s]$：跑完整实验、加独立观测噪声后取条件平均。

两者应在 Monte Carlo 误差内一致。注意：这里**不读取任何 simulator 的 outcome 真值公式来"算答案"**，只是验证识别出的 functional 与目标量相等。

## 5. Identification 输出什么，Estimation 接什么？

Identification 到式 (3) 就结束，输出四件事：

1. **target**：$\mu_i(t,s)$，$t\in\{0,1\}, s\in\{0,1\}$（本 exposure 下每节点 4 个 arm mean，$N=300$ 共 $4N=1200$ 个）；
2. **assumptions**：A1 处理前结构固定、A2 consistency、A3 full-vector randomization（来自协议）、A4 support（数值检查）；
3. **population support**：$\pi_i(t,s)=p_i^{\,t}(1-p_i)^{1-t}q_i(s)$，零 support arm = 孤立点的 $(t,1)$；
4. **POINT functional**：$\mu_i(t,s)=\mathbb{E}[Y_i^{obs}\mid A,X,T_i=t,S_i=s]$（support $>0$ 时）。

随后可形成三个对比量（留给 Estimation）：

$$\tau_i^{Direct}=\mu_i(1,0)-\mu_i(0,0),\qquad
\tau_i^{Spillover}=\mu_i(1,1)-\mu_i(1,0),\qquad
\tau_i^{Total}=\mu_i(1,1)-\mu_i(0,0).$$

有限实验中某个 arm 恰好没出现，只是 estimation 困难；只要 population support $>0$，不会把 POINT 改成 PARTIAL。

In [ ]:
import os, sys, itertools, random
sys.path.insert(0, os.path.abspath("."))   # 兼容从任意目录打开 notebook

from support import (er_network, check_probs, neighbor_treated_pmf,
                     exposure_probs, support_pi, support_table, zero_support_arms)
from rule import ASSUMPTIONS, identify, identify_all

# ---- 按任务书设定构造演示数据：N=300 的 ER 网络 + 实验前已知的异构 p_i ----
N = 300
neighbors = er_network(N, 5.0 / (N - 1), seed=42)          # 平均度约 5 的 ER 图
rng = random.Random(2026)
probs = [round(rng.uniform(0.05, 0.95), 4) for _ in range(N)]   # 0 < p_i < 1，实验前已知
check_probs(probs)                                          # 设计约束校验

degrees = [len(nb) for nb in neighbors]
isolated = [i for i, d in enumerate(degrees) if d == 0]
print(f"N = {N}, 平均度 = {sum(degrees)/N:.2f}, 最大度 = {max(degrees)}")
print(f"p_i: min = {min(probs)}, max = {max(probs)}（互不相同、实验前已知）")
print(f"孤立点 (d_i = 0): {isolated}")

In [ ]:
# ---- 单个节点的 support 表 + 全网络的零 support arm ----
i0 = 0
print(f"node {i0}: d_i = {degrees[i0]}, p_i = {probs[i0]}")
pmf = neighbor_treated_pmf(neighbors, probs, i0)
print(f"  P(K_i = k), k=0..{degrees[i0]}: " + ", ".join(f"{v:.4f}" for v in pmf))
q = exposure_probs(neighbors, probs, i0)
print(f"  q_i(0) = P(K_i <= {degrees[i0]//2}) = {q[0]:.4f},  q_i(1) = P(K_i > {degrees[i0]//2}) = {q[1]:.4f}")
tab = support_table(neighbors, probs, i0)
print("  support 表 (t,s) -> pi_i(t,s):")
for (t, s), v in tab.items():
    manual = (probs[i0] if t == 1 else 1 - probs[i0]) * q[s]
    print(f"    (t={t}, s={s}): pi = {v:.6f}   [P(T_i=t)*q(s) = {manual:.6f}, 合计归一 = {abs(sum(tab.values())-1) < 1e-12}]")

zsa = zero_support_arms(neighbors, probs)
print(f"\n全网络 zero-support arms: {sorted(zsa)}")
print(f"原因: {list(zsa.values())[0] if zsa else '无'}")

In [ ]:
# ---- D1 与 baseline 的实质差别：条件权重 w(t_{-i}) 不再均摊 ----
# 找一个度数小的节点，枚举其邻居的全部 2^{d_i} 种处理组合
i_small = next(i for i in range(N) if 2 <= degrees[i] <= 3)
nb = neighbors[i_small]
print(f"node {i_small}: 邻居 = {nb}, 对应 p = {[probs[j] for j in nb]}")
for s in (0, 1):
    rows = []
    for bits in itertools.product((0, 1), repeat=len(nb)):
        k = sum(bits)
        s_of = 1 if k > len(nb) // 2 else 0
        if s_of != s:
            continue
        p_design = 1.0
        for j, b in zip(nb, bits):
            p_design *= probs[j] if b else 1.0 - probs[j]
        rows.append((bits, p_design))
    z = sum(p for _, p in rows)
    print(f"\nS_i = {s}（{len(rows)} 种邻居组合）:")
    for bits, p in rows:
        print(f"    t_N(i) = {bits}: 设计概率 {p:.4f} -> 条件权重 w = {p/z:.4f}"
              f"   （baseline 均摊为 {1/len(rows):.4f}）")
print("\n=> D1 下 w 按各 p_j 倾斜；这只改变 Estimation 的权重，不改变识别结论。")

In [ ]:
# ---- IdentificationResult：非孤立点四个 arm 全 POINT；孤立点 (t,1) 被拒绝 ----
print(f"=== node {i0}（d_i = {degrees[i0]}，非孤立）===")
for (t, s), res in identify_all(neighbors, probs, i0).items():
    print(" ", res)

iso = isolated[0]
print(f"\n=== node {iso}（孤立点，d_i = 0）===")
for (t, s), res in identify_all(neighbors, probs, iso).items():
    print(" ", res)

# 查询计数
n_point = sum(1 for i in range(N) for t in (0, 1) for s in (0, 1)
              if identify(neighbors, probs, i, t, s).identified)
print(f"\n全网络: {4*N} 个 arm 中 {n_point} 个 POINT identified，{4*N-n_point} 个被拒绝（均为孤立点的 (t,1)）")

In [ ]:
# ---- 小图模拟验证 E[Y^obs | T_i=t, S_i=s] = mu_i(t,s) ----
N_sim, M = 30, 20000
nbrs_sim = er_network(N_sim, 4.0 / (N_sim - 1), seed=11)
rng2 = random.Random(7)
ps_sim = [round(rng2.uniform(0.10, 0.90), 4) for _ in range(N_sim)]
alpha = [rng2.gauss(0.0, 1.0) for _ in range(N_sim)]
BETA_OWN, BETA_SPILL = 0.8, 0.4

def potential_outcome(i, t, t_vec):
    k = sum(t_vec[j] for j in nbrs_sim[i])
    return alpha[i] + BETA_OWN * t + BETA_SPILL * k

def exposure_of(i, t_vec):
    k = sum(t_vec[j] for j in nbrs_sim[i])
    return 1 if k > len(nbrs_sim[i]) // 2 else 0

check_nodes = [i for i in range(N_sim) if 2 <= len(nbrs_sim[i]) <= 4][:3]
print(f"验证节点: {check_nodes}（度数 {[len(nbrs_sim[i]) for i in check_nodes]}），每格 M = {M} 次模拟\n")

def true_mu_all(i):
    acc = {(t, s): 0.0 for t in (0, 1) for s in (0, 1)}
    cnt = {(t, s): 0 for t in (0, 1) for s in (0, 1)}
    for _ in range(M):
        t_vec = [0] * N_sim
        for j in range(N_sim):
            if j != i:
                t_vec[j] = 1 if rng2.random() < ps_sim[j] else 0
        s_obs = exposure_of(i, t_vec)
        for t in (0, 1):
            acc[(t, s_obs)] += potential_outcome(i, t, t_vec)
            cnt[(t, s_obs)] += 1
    return {k: acc[k] / cnt[k] for k in acc}

def observed_mu_all(i):
    acc = {(t, s): 0.0 for t in (0, 1) for s in (0, 1)}
    cnt = {(t, s): 0 for t in (0, 1) for s in (0, 1)}
    for _ in range(M):
        t_vec = [1 if rng2.random() < ps_sim[j] else 0 for j in range(N_sim)]
        s_obs = exposure_of(i, t_vec)
        y_obs = potential_outcome(i, t_vec[i], t_vec) + rng2.gauss(0.0, 1.0)
        acc[(t_vec[i], s_obs)] += y_obs
        cnt[(t_vec[i], s_obs)] += 1
    return {k: acc[k] / cnt[k] for k in acc}, cnt

print(f"{'node':>4} {'(t,s)':>6} {'true mu':>9} {'observed':>9} {'|diff|':>8} {'n_cell':>7} {'tol':>7}")
all_ok = True
for i in check_nodes:
    tm = true_mu_all(i)
    om, cnt = observed_mu_all(i)
    for key in [(0, 0), (0, 1), (1, 0), (1, 1)]:
        t, s = key
        tol = 6.0 / (cnt[key] ** 0.5) + 0.02      # 观测噪声 + 真值两侧 MC 误差的宽松界
        diff = abs(tm[key] - om[key])
        all_ok = all_ok and diff < tol
        print(f"{i:>4} ({t},{s}) {tm[key]:>9.4f} {om[key]:>9.4f} {diff:>8.4f} {cnt[key]:>7} {tol:>7.4f}")
assert all_ok, "式 (3) 数值验证失败"
print("\n[ok] 式 (3) 数值验证通过：观测条件均值 = design-standardized arm mean")

In [ ]:
# ---- 总结 ----
print("任务 01（D1 + E1）框架小结")
print("=" * 56)
print(f"随机化设计 : D1  T_i ~ Bernoulli(p_i) 独立, 0 < p_i < 1, p_i 已知")
print(f"exposure   : E1  S_i = 1{{K_i > floor(d_i/2)}}（严格多数，同 baseline）")
print(f"support    : pi_i(t,s) = p_i^t (1-p_i)^(1-t) * q_i(s),  K_i ~ PoissonBinomial")
print(f"POINT      : mu_i(t,s) = E[Y^obs_i | A, X, T_i = t, S_i = s]   (pi_i(t,s) > 0)")
print(f"拒绝       : pi_i(t,s) = 0 的 arm（本网络中 {4*N-n_point} 个，均为孤立点 (t,1)）")
print(f"验证       : test_rule.py 中 N=7 brute-force 枚举 2^7 个 assignment 对拍一致")